# Rezidualne neuronske mreže

<center><img src="Images/V11_banner.png" width="700" height="700"/></center>



### Do sada viđeno

U prethodnom poglavlju opisana je konvolucijska neuronska mreža gdje je pokazano da filteri naučeni konvolucijom mogu postići zapažene rezultate kada je ulazni podatak slika. Također je pokazano da neuronska mreža mora imati dovoljan broj filtera, odnosno imati dovoljan kapacitet. Međutim već smo vidjeli da takve duboke neuronske mreže ne mogu ići u nedogled sa brojem slojeva zbog nestajućih gradijenata. 

Stoga u ovoj vježbi će se predstaviti **rezidualni blokovi** inicijalno predloženi 2015 godine u ResNet neuronskoj mreži. Više o samoj mreži i natjecanju ILSVRC može se pronaći na idućoj [poveznici](https://arxiv.org/pdf/1512.03385).

---

### Sekvencijalne neuronske mreže 

Do sada smo uvijek promatrali sekvencijalne neuronske mreže,što bi u prijevodu značilo da svaki svaki sloj prima izlaz prethodnog sloja i prosljeđuje rezultat sljedećem sloju, kao što je prikazano na idućoj slici:



<center><img src="Images/V11_SEQ.png" width="700" height="700"/></center>

Odnosno ako bi se ista raspisala pomoću jednadžbi:


\begin{aligned}
h_1 &= f_1[x, \phi_1] \\
h_2 &= f_2[h_1, \phi_2] \\
h_3 &= f_3[h_2, \phi_3] \\
y   &= f_4[h_3, \phi_4]
\end{aligned}

odnsosno kao jedna jednadžba koja predstavlja niz ugnježđenih funkcija:

\begin{align}
y = f_4\Big(
      f_3\big(
        f_2( f_1[x, \phi_1], \phi_2 ),
        \phi_3
      ),
      \phi_4
    \Big)
\end{align}

U jednadžbama $h_1$, $h_2$ i $h_3$ predstavalju međurezultate skrivenih slojeva, x je ulaz mreže, a y je izlaz. Funkcije $f[\cdot, \phi_k]$ obavljaju obradu podataka u vidu, primjerice, aktivacijskih funkcija. U standardnoj neuronskoj mreži, svaki se sloj sastoji od linearne transformacije praćene aktivacijskom funkcijom, a parametri $\phi_k$ obuhvaćaju težine i pomake (bias).

U principu, može se nanizati na ovaj način bilo koliko slojeva, ali u jednom trenutku, kako je pokazano u prethodnim vježbama, performanse neuronske mreže se smanjuju kreiranjem pre-dubokih neuronskih mreža (vježba 8). Razlog tomu su bili nestajći gradijenti koji se do sada adresirao incijalizacijom parametara na razumni raspon. Međutim, zbog velikog broja slojeva u dubokim neuronskim mežama, mala promjena ulaza izrazito utječe na gradijent i rezultira potpuno drugačijim gradijentom. Naime, funkcija cilja nije glatka krivulja već je popu planina i dolina, stoga uz postojanu stopu učenja vrlo lako se može preskočiti određena planina. Ovaj fenomen se naziva fenomen **razbijenih gradijenata** (*engl. shattered gradients*). 

Ovaj problem se javlja zato što promjene u ranim slojevima mreže mijenjaju izlaz na sve složeniji način kako mreža postaje dublja. Derivacija izlaza $y$ u odnosu na prvi sloj mreže u jednadžbi jest sljedeća:

\begin{align}
\frac{\partial y}{\partial f_1}
= \frac{\partial f_2}{\partial f_1}
  \frac{\partial f_3}{\partial f_2}
  \frac{\partial f_4}{\partial f_3}
\end{align}

Kada se parametri koji definiraju $f_1$ promjene, sve derivacije u ovom nizu evaluiraju se na blago različitim točkama jer se slojevi
$f_2$, $f_3$ i $f_4$ računaju na temelju izlaza funkcije $f_1$. Posljedično, ažurirani gradijent u svakoj iteraciji treniranja može biti potpuno drugačiji, a funkcija gubitka postaje nestabilna. Kako bi se uklonio ovaj problem omogućilo treniranje dubokih mreža, koriste se **residualni blokovi** i **"preskočne veze"**

---

## Rezidualni blokovi i preskočne veze

Rezidualne ili *skip veze* su grane u računskoj putanji u kojima se ulaz svakog sloja mreže pribraja njegovom izlazu, odnosno kao što je to vidljivo na slici:

<center><img src="Images/V11_NN.png" width="700" height="700"/></center>

Ako izrazimo ovakvu neuronsku mrežu pomoću formula, dobivamo sljedeći izraz:


\begin{aligned}
h_1 &= x + f_1[x, \phi_1] \\
h_2 &= h_1 + f_2[h_1, \phi_2] \\
h_3 &= h_2 + f_3[h_2, \phi_3] \\
y   &= h_3 + f_4[h_3, \phi_4]
\end{aligned}

gdje je prvi član na desnoj strani svake jednadžbe rezidualna veza. Svaka funkcija $f_i$ uči aditivnu promjenu trenutne reprezentacije. Iz toga slijedi da njihovi izlazi moraju biti iste veličine kao i njihovi ulazi. Svaka aditivna kombinacija ulaza i obrađenog izlaza naziva se **rezidualni blok** ili rezidualni sloj. Ukoliko se raspišu skriveni slojevi $h_i$, dani izraz može se raspisati kao jedna "velika" funkcija:

\begin{aligned}
y =\;& x + f_1[x] \\
&+ f_2[x + f_1[x]] \\
&+ f_3\big[x + f_1[x] + f_2[x + f_1[x]]\big] \\
&+ f_4\big[x + f_1[x] + f_2[x + f_1[x]] + f_3[x + f_1[x] + f_2[x + f_1[x]]]\big]
\end{aligned}

pri čemu su u formulama parametri $\phi_i$ izbačeni zbog jasnoće. Jasno je prikazano da je posljednji izlaz neuronske mreće zapravo suma ulaza i *četiri manje neuronske mreže*, pa se stoga rezidualna mreža može promatrati i kao spoj manjih nuronskih mreža čiji su izlazi sumirani u jedan konkretni rezultat. 

---
<font color='red'>


## Zadatak

<left><img src="Images/Zadatak.png" width="70" height="70"/></left>
</font>

Za prethodno zadanu neuronsku mrežu postoji prethodno danu neuronsku mrežu postoji $16$ različitih putova: Navedite ih i odgovorite u koliko od tih putova sudjeluje $f_1$?

---

## Redoslijed operacija u rezidualnom bloku

Do sada se podrazumijevalo da je aditivna funkcija $f[x]$ mogla biti bilo koji validan sloj mreže (npr. potpuno povezani ili konvolucijski). Tehnički je to točno, ali valja napomenuti da je redoslijed operacija u tim funkcijama i dalje važan. Ako ne sadrže nelinearnu aktivacijsku funkciju poput ReLU, cijela mreža bit će linearna. Međutim, u tipičnom sloju mreže, ReLU funkcija nalazi se na kraju, pa je izlaz nenegativan. Ako se usvoji ova konvencija, tada svaki rezidualni blok može samo povećavati vrijednosti ulaza.

Stoga je uobičajeno promijeniti redoslijed operacija tako da se najprije primijeni aktivacijska funkcija, a zatim linearna transformacija. Ponekad može postojati više slojeva obrade unutar pojedinog bloka, ali se obično primjenjuje barem jedna linearna transformacija. Konačno, mreže s ReLU operacijom neće učiniti ništa od početnog ulaza mreže ako je on negativan, jer će ReLU odsjeći cijeli signal na nulu. Stoga je uobičajeno započeti mrežu linearnom transformacijom, a ne rezidualnim blokom.

<center><img src="Images/V11_Order.png" width="500" height="500"/></center>

---

## Problem eksplodirajućih gradijenata
U prethodnim vježbama pokazalo se da je **inicijalizacija parametara mreže** ključna. Bez pažljive inicijalizacije, magnitude međuvrijednosti tijekom prolaza unaprijed u algoritmu povratne propagacije (*backpropagation*) mogu eksponencijalno rasti ili opadati. Slično tome, gradijenti tijekom prolaza unatrag mogu eksplodirati ili nestajati kako se pogreška kreće unatrag kroz mrežu.

Stoga se parametri mreže inicijaliziraju tako da **očekivana varijanca aktivacija** (u prolazu unaprijed) i **gradijenata** (u prolazu unatrag) ostaje ista između slojeva. Prisjetimo se: **He inicijalizacija** to postiže za ReLU aktivacije tako da inicijalizira pomake (*bias*) $\beta$ na nulu i bira normalno distribuirane težine $\Omega$ s očekivanjem nula i varijancom

$$
\frac{2}{D_h},
$$

gdje je $D_h$ broj skrivenih neurona u prethodnom sloju.

U slučaju **rezidualne mreže** nestajanje međuvrijednosti ili gradijenata s dubinom mreže nije više problem jer postoji putanja kojom svaki sloj izravno doprinosi izlazu mreže. Međutim, čak i ako se koristi He inicijalizacija unutar rezidualnog bloka, vrijednosti u prolazu unaprijed rastu eksponencijalno kako informacija putuje dalje kroz mrežu.

Razlog tome je što se rezultat obrade u rezidualnom bloku **pribraja ulazu**. Svaka grana ima određenu (nepovezanu) varijabilnost, pa se ukupna varijanca povećava kada se one ponovno kombiniraju. Uz ReLU aktivacije i He inicijalizaciju, očekivana varijanca se **ne mijenja** obradom unutar svakog pojedinog bloka. Posljedično, kada se izlaz bloka ponovno kombinira s ulazom, varijanca se **udvostručuje**  te raste eksponencijalno s brojem rezidualnih blokova.

To ograničava maksimalnu dubinu mreže prije nego što se u prolazu unaprijed premaši preciznost brojeva s pomičnim zarezom (*floating-point precision*). Sličan argument vrijedi i za gradijente u prolazu unatrag algoritma povratnog širenja.

Stoga rezidualne mreže i dalje pate od **nestabilne propagacije unaprijed** i **eksplodirajućih gradijenata**, čak i uz He inicijalizaciju. Jedan pristup stabilizaciji prolaza unaprijed i unatrag bio bi koristiti He inicijalizaciju, a zatim pomnožiti kombininirani izlaz svakog rezidualnog bloka s

$$
\frac{1}{\sqrt{2}},
$$

kako bi se kompenziralo udvostručavanje varijance. Međutim, u praksi je znatno češće koristiti **batch normalizaciju**.

---

## Batch normalizacija

**Batch normalizacija** (*Batch Normalization* ili **BatchNorm**) pomiče i skalira svaku aktivaciju $(h)$ tako da njezina **srednja vrijednost** i **varijanca** preko mini-skupa (batcha) $(\mathcal{B})$ postanu vrijednosti koje se **uče tijekom treniranja**. Najprije se izračunaju empirijska srednja vrijednost $(m_h)$ i standardna devijacija $(s_h)$:

$$
m_h = \frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} h_i
$$

$$
s_h = \sqrt{\frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} (h_i - m_h)^2}
$$

gdje su sve veličine skalarne. Zatim se ove statistike koriste za **standardizaciju** aktivacija u batchu tako da imaju srednju vrijednost nula i jediničnu varijancu:

$$
h_i \leftarrow \frac{h_i - m_h}{s_h + \epsilon},
\qquad \forall i \in \mathcal{B}
$$

gdje je $(\epsilon)$ mali broj koji sprječava dijeljenje s nulom u slučaju da je $(h_i)$ isto za sve elemente batcha pa je $(s_h = 0)$.

Na kraju, normalizirana varijabla se skalira s $(\gamma)$ i pomiče za $(\delta$):

$$
h_i \leftarrow \gamma h_i + \delta,
\qquad \forall i \in \mathcal{B}
$$

Nakon ove operacije, aktivacije imaju srednju vrijednost $(\delta)$ i standardnu devijaciju $(\gamma)$ preko svih članova batcha. Obje ove veličine uče se tijekom treniranja.

Batch normalizacija se primjenjuje **neovisno na svaku skriveni neuron**. U standardnoj neuronskoj mreži s $(K)$ slojeva, od kojih svaki sadrži $(D)$ skrivenih neurona, postoji $(K D)$ naučenih pomaka $(\delta)$ i $(K D)$ naučenih skala $(\gamma)$. U konvolucijskoj mreži normalizacija se primjenjuje po **kanalima**. Ako mreža ima $(K)$ slojeva, svaki s $(C)$ kanala, tada postoje $(K C)$ pomaka $(\delta)$ i $(K C)$ skala $(\gamma)$.

U fazi testiranja ne postoji batch iz kojeg bi se mogla izračunati statistika. Stoga se statistike $(m_h)$ i $(s_h)$ izračunavaju se preko **cijelog skupa za treniranje** (umjesto preko pojedinog batcha) te se zatim **zamrzavaju** u konačnoj mreži.

Postoji još nekoliko verzija normalizacija od kojih svaka ima svoje prednosti, ali i mane. Više se može saznati na [poveznici#1](https://medium.com/syncedreview/facebook-ai-proposes-group-normalization-alternative-to-batch-normalization-fb0699bffae7), odnosno [poveznici#2](https://theaisummer.com/normalization/).


Pozicija batch normalizacije je prije aktivacijskih funkcija:

<center><img src="Images/V11_skip.png" width="700" height="700"/></center>

### Prednosti Batch normalizacije (BN)

Batch normalizacija ima nekoliko važnih prednosti:

- **Ubrzava treniranje dubokih neuronskih mreža.**

- Za svaki ulazni mini-batch izračunavaju se različite statistike, čime se uvodi određeni oblik **regularizacije**. Regularizacija se odnosi na bilo koju tehniku ili ograničenje kojim se tijekom treniranja smanjuje složenost duboke neuronske mreže.

- Svaki mini-batch ima vlastitu distribuciju aktivacija. Promjena između tih mini-distribucija naziva se **interni kovarijatni pomak** (*Internal Covariate Shift*). U početku se smatralo da batch normalizacija eliminira ovaj fenomen, no Santurkar i sur. [7] kasnije su pokazali da to nije stvarni razlog zbog kojeg BN funkcionira.

- Batch normalizacija ima povoljan učinak na **tok gradijenata** kroz mrežu. Smanjuje ovisnost gradijenata o skali parametara ili o njihovim početnim vrijednostima, što omogućuje korištenje znatno većih **learning rate** vrijednosti.

- U teoriji, batch normalizacija omogućuje korištenje **zasićenih nelinearnih aktivacijskih funkcija** jer sprječava da mreža zapne u lošim režimima učenja. U praksi se, međutim, ove vrste aktivacijskih funkcija gotovo nikada ne koriste.

---

### Nedostaci Batch normalizacije

Batch normalizacija također ima i određene nedostatke:

- **Netočna procjena batch statistika** pri malim veličinama batcha, što povećava pogrešku modela. U zadacima kao što su predikcija videa, segmentacija i obrada 3D medicinskih slika batch veličina je često vrlo mala. Batch normalizacija zahtijeva dovoljno velik batch.

- **Problemi kada se batch veličina mijenja**. Tipični primjeri uključuju razlike između:
  - treniranja i inferencije,
  - predtreniranja i finog podešavanja (*fine-tuning*),
  - backbone arhitekture i pripadnog head dijela mreže.

---

## Popularne topologije ResNet i Densnet

# Usporedba **DenseNet** i **ResNet** arhitektura dubokih neuronskih mreža

Ovaj pregled namijenjen je studentima kao sažeta, ali konceptualno jasna usporedba dviju utjecajnih arhitektura dubokog učenja: **Residual Networks (ResNet)** i **Densely Connected Convolutional Networks (DenseNet)**. Obje arhitekture razvijene su s ciljem rješavanja problema treniranja vrlo dubokih neuronskih mreža.

---

## Residual Networks (ResNet)
K. He, X. Zhang, S. Ren, J. Sun, CVPR 2016, [https://arxiv.org/abs/1512.03385](https://arxiv.org/abs/1512.03385)


### Osnovna ideja
ResNet uvodi **rezidualne (skip) veze** koje omogućuju da se izlaz nekog sloja izravno zbraja s izlazom dubljeg sloja. Time se uči **rezidualna funkcija**, a ne izravna preslikavanja.

Rezidualni blok konceptualno uči:

\begin{align}
\mathbf{y} = F(\mathbf{x}) + \mathbf{x}
\end{align}

gdje je \(F(\mathbf{x})\) funkcija koju uče slojevi unutar bloka.

### Ključne značajke
- Skip veze između blokova
- Ublažavanje problema **nestajanja gradijenata**
- Omogućuje treniranje vrlo dubokih mreža (100+ slojeva)
- Široko korišten standard u industriji i istraživanju

### Prednosti
- Stabilno treniranje dubokih mreža  
- Dobra skalabilnost dubine  
- Jednostavna i robusna arhitektura  
- Izvrsni empirijski rezultati na velikim skupovima podataka  

### Nedostaci
- Moguća redundancija značajki  
- Manje eksplicitno poticanje ponovne uporabe značajki  
- Veća potreba za parametrima u usporedbi s DenseNetom  

---

## Densely Connected Networks (DenseNet)
G. Huang, Z. Liu, L. van der Maaten, K. Q. Weinberger, CVPR 2017 ,[https://arxiv.org/abs/1608.06993](https://arxiv.org/abs/1608.06993)

### Osnovna ideja
DenseNet povezuje **svaki sloj sa svim prethodnim slojevima** unutar istog bloka. Ulazi slojeva se **konkateniraju**, a ne zbrajaju.

Za $(l)$-ti sloj vrijedi:

\begin{align}
\mathbf{x}_l = H_l([\mathbf{x}_0, \mathbf{x}_1, \dots, \mathbf{x}_{l-1}])
\end{align}

### Ključne značajke
- Gusta povezanost slojeva (dense connectivity)
- Eksplicitna ponovna uporaba značajki
- Manji broj parametara
- Bolji protok gradijenata kroz mrežu

### Prednosti
- Vrlo efikasna uporaba parametara  
- Manje pretreniravanje (implicitna regularizacija)  
- Jači protok informacija i gradijenata  
- Često bolja generalizacija na manjim skupovima podataka  

### Nedostaci
- Veći memorijski zahtjevi (zbog konkatenacije značajki)  
- Složenija implementacija  
- Sporije izvođenje za vrlo duboke mreže  

### Izvorni rad
- **:contentReference[oaicite:1]{index=1}**  


---

## Izravna usporedba

| Značajka | ResNet | DenseNet |
|--------|--------|----------|
| Tip veze | Zbrajanje (additive skip) | Konkatenacija |
| Povezanost slojeva | Samo s prethodnim blokom | Sa svim prethodnim slojevima |
| Uporaba značajki | Implicitna | Eksplicitna |
| Broj parametara | Veći | Manji |
| Memorijska potrošnja | Umjerena | Veća |
| Dubina mreže | Vrlo velika (100+) | Umjerena do velika |
| Jednostavnost implementacije | Visoka | Srednja |
| Tipične primjene | Opći standard (CV) | Manji skupovi, medicinsko snimanje |

---

## Sažetak

Obje arhitekture predstavljaju ključne prekretnice u razvoju dubokog učenja. ResNet je postao de facto standard za duboke mreže, dok DenseNet pruža elegantno rješenje za efikasno učenje i bolju generalizaciju, osobito u domenama s ograničenom količinom podataka. ResNet se može interpretirati kao mreža koja *uči korekcije* prethodnih reprezentacija. DenseNet se može interpretirati kao *akumulacija znanja*, gdje svaki sloj vidi sve prethodno naučeno.


---

<font color='red'>


## Zadatak

<left><img src="Images/Zadatak.png" width="70" height="70"/></left>
</font>

Isprobajte sljedeći programski kod te uočite prednosti preskočnih veza te rezidualnih blokova. Proučite programski kod te pokušajte isprintati arhitekturu po uzoru na prethodne vježbe. Iznesite svoje zaključke!

In [ ]:
from Skripte.Vjezba11.widget import unified_mlp_conv2d_widget
from IPython.display import display

# Autoreload
%load_ext autoreload
%autoreload 2

display(unified_mlp_conv2d_widget())